# Exercise 10: The Min-K% Membership Inference Attack

A **membership inference attack (MIA)** tries to tell whether a specific piece of text was part of a model's training data. The intuition: a causal LM tends to assign higher probability to tokens it has already memorized. Min-K% (Shi et al., 2023) turns this intuition into a concrete score.

In [ ]:
import torch

def get_token_probs(sentence: str, model, tokenizer) -> list[float]:
    """
    Returns the probability the model assigned to each ground-truth
    next token in `sentence`.
    """

    # tokenize the sentence and run it through the model
    inputs = tokenizer(sentence, return_tensors="pt")
    input_ids = inputs.input_ids[0]
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    logits = outputs.logits[0] 

    # shift logits/labels so position i predicts token i+1
    shift_logits = logits[:-1, :]
    shift_labels = input_ids[1:]

    # softmax the shifted logits into probabilities
    probs = torch.softmax(shift_logits, dim=-1)

    # gather the probability of each actual next token
    token_probs = []
    
    for i in range(len(shift_labels)):
        target_token_id = shift_labels[i]
        # Extract the probability the model assigned to the actual next token
        prob = probs[i, target_token_id].item()
        token_probs.append(prob)

    return token_probs

## Task 1: The Min-K% score

A member document doesn't necessarily have a high probability on *every* token — a rare name or an unusual phrasing can still look surprising even to a model that memorized the passage. Shi et al. (2023) found that looking only at a text's *least likely* tokens (the bottom $k\%$) gives a stronger membership signal than averaging over all of them, since it's exactly on these hardest tokens that a memorized document still looks comparatively confident, while a genuinely unseen document does not.

$$\text{Min-K\%}(x) = \frac{1}{|\text{bottom-}k\%|}\sum_{t\, \in\, \text{bottom-}k\%} p(t)$$

(Summing raw probabilities rather than the paper's log-likelihood is fine for our purposes — AUROC only depends on the *ranking* of scores, not their absolute scale.)

**Task:**

1. Call your `get_token_probs` from Task 1.
2. Sort the probabilities ascending.
3. Keep only the lowest `k_percent` fraction.
4. Return their sum as the Min-K% score. Higher score $\Rightarrow$ the model finds the text more "expected" $\Rightarrow$ more likely to be a member.

In [ ]:
def min_k(sentence: str, model, tokenizer, k_percent: float = 0.20) -> float:
    token_probs = get_token_probs(sentence, model, tokenizer)

    # TODO (a): sort token_probs ascending

    # TODO (b): keep only the lowest k_percent fraction

    # TODO (c): return the aggregated score

# Example usage:
# min_k("The Ruhr-University Bochum is located in Germany.", model, tokenizer, k_percent=0.20)

## Task 3: Evaluating the attack on WikiMIA

To know whether Min-K% actually works, we need labeled data: sentences the target model *did* see during training (members) and sentences it could not have seen, e.g. published after its training cutoff (non-members). WikiMIA (Shi et al., 2023) is exactly this kind of benchmark. Because `min_k` produces an unbounded, uncalibrated score rather than a probability, we can't threshold it directly — instead we use AUROC, which only checks whether member scores tend to rank above non-member scores.

**Task:**

1. Load a WikiMIA split, e.g. `datasets.load_dataset("swj0419/WikiMIA", split="WikiMIA_length64")`.
2. For every sample, record its true `label` and compute `min_k(sample["input"], model, tokenizer)`.
3. Compute `roc_auc_score(y_true, y_score)`.
4. An AUROC around $0.5$ means the attack is no better than a coin flip; closer to $1.0$ means members and non-members are well separated. Try a few different `k_percent` values (e.g. $0.05, 0.2, 0.5$) — which works best, and why might that be?

In [ ]:
from datasets import load_dataset
from sklearn.metrics import roc_auc_score

# TODO (1): load a WikiMIA split
dataset = ...

# TODO (2): compute the Min-K% score for `text`


## Task 4: Why is WikiMIA not suited ?
WikiMIA was constructed by declaring training samples before the cutoff-date as members and after that as non-members. Can you explain, why this is bad for evaluating Membership Infernce Attacks ?.

## Task 5: Blind Baseline
To validate REDACTED, one can employ a simple model-free bag-of-word classifier as shown in the works by Das et al. 2025 (BLIND BASELINES BEAT MEMBERSHIP INFERENCE ATTACKS FOR FOUNDATION MODELS). Implement bag-of-word classifier, by using `from sklearn.feature_extraction.text import CountVectorizer` for constructing the bag-of-word and `from sklearn.naive_bayes import MultinomialNB` to predict the membership label. 